## **SQL Query Generation from Natural Language using PEFT and LoRA**


## **1. Environment Setup**

In [1]:
# install the required libraries
!pip install -q \
  "transformers==5.0.0" \
  "peft==0.18.1" \
  "accelerate==1.13.0" \
  "datasets==4.8.4" \
  "trl==1.1.0" \
  "sentencepiece==0.2.1" \
  "protobuf==5.29.6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.5 MB/s eta 0:00:00


In [2]:
import sys
import torch # pre-installed in colab env

print("Python     :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)

# check GPU availability
print("CUDA avail :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name   :", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory : {props.total_memory / 1024**3:.2f} GB")

# Expected on Colab free tier: Tesla T4, ~14.5 GB

Python     : 3.12.13
PyTorch      : 2.11.0+cu128
CUDA avail : True
GPU name   : Tesla T4
GPU memory : 14.56 GB


## **2. Load the base model in FP16**

We load the model in half precision (`torch.float16`). On T4 this is the sweet spot — faster than FP32

> Add blockquote



In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False          # required for gradient checkpointing later
model.config.pretraining_tp = 1

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params/1e6:.1f} M")
print(f"Memory footprint: {model.get_memory_footprint()/1024**3:.2f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Total parameters: 1100.0 M
Memory footprint: 2.05 GB


## **3. Prompt Design and Base Model Evaluation**

The TinyLlama chat template is used to structure inputs. The model receives a schema and a user question and generates the corresponding SQL query.

In [4]:
def build_prompt(schema: str, question: str) -> str:
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    user = f"Schema:\n{schema}\n\nQuestion: {question}"
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [5]:
# testing build_prompt function
test_prompt_1 = build_prompt(
    schema="CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
    question="List the names of employees in the Engineering department earning more than 100000."
)

test_prompt_1

'<|system|>\nYou are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.</s>\n<|user|>\nSchema:\nCREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);\n\nQuestion: List the names of employees in the Engineering department earning more than 100000.</s>\n<|assistant|>\n'

In [6]:
print(test_prompt_1)

<|system|>
You are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.</s>
<|user|>
Schema:
CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);

Question: List the names of employees in the Engineering department earning more than 100000.</s>
<|assistant|>



In [8]:
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 120) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    # Slice only newly generated tokens
    input_length = inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][input_length:]

    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


In [9]:
output = generate(test_prompt_1)
print(output)

To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE department = 'Engineering'
AND salary > 100000;
```

This query will return a list of names of employees in the Engineering department who have a salary greater than 100,000.


In [10]:
# Three probe prompts we will reuse AFTER fine-tuning for direct comparison.
PROBES = [
    {
        "schema":  "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary INT);",
        "question":"List the names of employees in the Engineering department earning more than 100000.",
    },
    {
        "schema":  "CREATE TABLE orders (order_id INT, customer_id INT, amount FLOAT, order_date DATE);",
        "question":"What is the total order amount per customer in 2024?",
    },
    {
        "schema":  "CREATE TABLE movies (title TEXT, year INT, rating FLOAT, genre TEXT);",
        "question":"Show the top 5 highest rated horror movies released after 2015.",
    },
]

print("="*70)
print("BASE MODEL (before fine-tuning)")
print("="*70)
base_outputs = []
for i, p in enumerate(PROBES, 1):
    prompt = build_prompt(p["schema"], p["question"])
    ans = generate(prompt)
    base_outputs.append(ans)
    print(f"\n--- Probe {i} ---")
    print("Q:", p["question"])
    print("A:", ans)


BASE MODEL (before fine-tuning)

--- Probe 1 ---
Q: List the names of employees in the Engineering department earning more than 100000.
A: To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE department = 'Engineering'
AND salary > 100000;
```

This query will return a list of names of employees in the Engineering department who have a salary greater than 100,000.

--- Probe 2 ---
Q: What is the total order amount per customer in 2024?
A: To answer the question, you need to use the `SUM` function to calculate the total order amount per customer in 2024. Here's the SQL query:

```sql
SELECT SUM(amount) AS total_order_amount_per_customer
FROM orders
WHERE year(order_date) = 2024
GROUP BY customer_id;
```

This query will return a single row with the total order amount per customer in 2024.

--- Probe 3 ---
Q: Show the top 5 highest rated horror movies released after 2015.
A: To show the top 5 highest rated horror movies released after 2015, 

The base model is evaluated before fine-tuning to establish a performance baseline.

## **4. Load and prepare the dataset**

The sql-create-context dataset is used for Text-to-SQL training on a small data subset.

In [11]:
from datasets import load_dataset

raw = load_dataset("b-mc2/sql-create-context", split="train")
print("Full dataset size:", len(raw))
print("Example row     :", raw[0])

# Keep it small for a fast, visible demo on T4
raw = raw.shuffle(seed=42).select(range(3000))
split = raw.train_test_split(test_size=0.05, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print("Train:", len(train_ds), " Eval:", len(eval_ds))

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json:   0%|          | 0.00/21.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Full dataset size: 78577
Example row     : {'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}
Train: 2850  Eval: 150


In [12]:
train_ds[0]

{'answer': 'SELECT first_name, last_name, hire_date FROM employees WHERE department_id = (SELECT department_id FROM employees WHERE first_name = "Clara")',
 'question': 'display the employee name ( first name and last name ) and hire date for all employees in the same department as Clara.',
 'context': 'CREATE TABLE employees (first_name VARCHAR, last_name VARCHAR, hire_date VARCHAR, department_id VARCHAR)'}

In [13]:
train_ds[:3]

{'answer': ['SELECT first_name, last_name, hire_date FROM employees WHERE department_id = (SELECT department_id FROM employees WHERE first_name = "Clara")',
  'SELECT longitude FROM table_16799784_8 WHERE name = "Razia Patera"',
  'SELECT opponent FROM table_21091145_1 WHERE black_knights_points = 27'],
 'question': ['display the employee name ( first name and last name ) and hire date for all employees in the same department as Clara.',
  'What is the longitude of the feature named Razia Patera? ',
  'Name the opponent for black knights points 27'],
 'context': ['CREATE TABLE employees (first_name VARCHAR, last_name VARCHAR, hire_date VARCHAR, department_id VARCHAR)',
  'CREATE TABLE table_16799784_8 (longitude VARCHAR, name VARCHAR)',
  'CREATE TABLE table_21091145_1 (opponent VARCHAR, black_knights_points VARCHAR)']}

In [14]:
print("Schema:", train_ds[0]["context"])
print("="*70)
print("Question:", train_ds[0]["question"])
print("="*70)
print("Answer:", train_ds[0]["answer"])
print("="*70)

Schema: CREATE TABLE employees (first_name VARCHAR, last_name VARCHAR, hire_date VARCHAR, department_id VARCHAR)
Question: display the employee name ( first name and last name ) and hire date for all employees in the same department as Clara.
Answer: SELECT first_name, last_name, hire_date FROM employees WHERE department_id = (SELECT department_id FROM employees WHERE first_name = "Clara")


In [15]:
def format_example(row):
    system = (
        "You are a SQL assistant. Given a table schema and a question, "
        "reply with ONLY the SQL query, nothing else."
    )
    user      = f"Schema:\n{row['context']}\n\nQuestion: {row['question']}"
    assistant = row["answer"]
    messages = [
        {"role": "system",    "content": system},
        {"role": "user",      "content": user},
        {"role": "assistant", "content": assistant},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

train_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_ds  = eval_ds.map (format_example, remove_columns=eval_ds.column_names)

print("\n--- Formatted training example ---\n")
print(train_ds[0]["text"][:800])# underatand the dataset
train_ds[0].keys()

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]


--- Formatted training example ---

<|system|>
You are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.</s>
<|user|>
Schema:
CREATE TABLE employees (first_name VARCHAR, last_name VARCHAR, hire_date VARCHAR, department_id VARCHAR)

Question: display the employee name ( first name and last name ) and hire date for all employees in the same department as Clara.</s>
<|assistant|>
SELECT first_name, last_name, hire_date FROM employees WHERE department_id = (SELECT department_id FROM employees WHERE first_name = "Clara")</s>



dict_keys(['text'])

In [16]:
print(train_ds[0])

{'text': '<|system|>\nYou are a SQL assistant. Given a table schema and a question, reply with ONLY the SQL query, nothing else.</s>\n<|user|>\nSchema:\nCREATE TABLE employees (first_name VARCHAR, last_name VARCHAR, hire_date VARCHAR, department_id VARCHAR)\n\nQuestion: display the employee name ( first name and last name ) and hire date for all employees in the same department as Clara.</s>\n<|assistant|>\nSELECT first_name, last_name, hire_date FROM employees WHERE department_id = (SELECT department_id FROM employees WHERE first_name = "Clara")</s>\n'}


## **5. Attach LoRA Adapters**

LoRA enables efficient fine-tuning by training only selected layers while keeping the base model unchanged.

In [17]:
from peft import LoraConfig, get_peft_model

# Enable gradient checkpointing to save VRAM during backward pass
model.gradient_checkpointing_enable()
model.enable_input_require_grads()     # needed because base params are frozen

lora_config = LoraConfig(
    r=16,                                # rank
    lora_alpha=32,                       # scaling factor (alpha / r = 2.0)
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# You should see something like: trainable: ~4M / all: ~1.1B  (< 0.5%)

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


## **6. Train**

Training is performed using SFTTrainer with settings optimized for the T4 GPU.


In [18]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./tinyllama-sql-lora"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    report_to="none",
)

# Pre-tokenize so we don't depend on SFTTrainer's text-field handling
def tokenize(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )
    return out

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_tok  = eval_ds.map (tokenize, batched=True, remove_columns=["text"])

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    processing_class=tokenizer,
)

trainer.train()

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
50,0.632610,0.619859
100,0.593653,0.577306
150,0.563651,0.566806


TrainOutput(global_step=179, training_loss=0.7018011961569334, metrics={'train_runtime': 475.2362, 'train_samples_per_second': 5.997, 'train_steps_per_second': 0.377, 'total_flos': 2464355322667008.0, 'train_loss': 0.7018011961569334})

In [19]:
# Peak VRAM used during training — should stay under T4's 15.8 GB
import torch
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

Peak GPU memory allocated: 2.39 GB


## Training Configuration

- **Epochs:** 1
- **Batch Size:** 2
- **Learning Rate:** 2e-4
- **FP16:** Enabled
- **Gradient Accumulation:** 8

## **7. Compare: AFTER Fine-Tuning**

In [20]:
# Re-enable cache for fast inference
model.config.use_cache = True
model.eval()

print("="*70)
print("FINE-TUNED MODEL (after LoRA)")
print("="*70)
for i, p in enumerate(PROBES, 1):
    prompt = build_prompt(p["schema"], p["question"])
    ans = generate(prompt)
    print(f"\n--- Probe {i} ---")
    print("Q:     ", p["question"])
    print("BEFORE:", base_outputs[i-1])
    print("="*70)
    print("AFTER :", ans)
    print("="*70)


FINE-TUNED MODEL (after LoRA)

--- Probe 1 ---
Q:      List the names of employees in the Engineering department earning more than 100000.
BEFORE: To answer this question, you can use the following SQL query:

```
SELECT name
FROM employees
WHERE department = 'Engineering'
AND salary > 100000;
```

This query will return a list of names of employees in the Engineering department who have a salary greater than 100,000.
AFTER : SELECT name FROM employees WHERE department = "engineering" AND salary > 100000

--- Probe 2 ---
Q:      What is the total order amount per customer in 2024?
BEFORE: To answer the question, you need to use the `SUM` function to calculate the total order amount per customer in 2024. Here's the SQL query:

```sql
SELECT SUM(amount) AS total_order_amount_per_customer
FROM orders
WHERE year(order_date) = 2024
GROUP BY customer_id;
```

This query will return a single row with the total order amount per customer in 2024.
AFTER : SELECT SUM(amount) FROM orders WHERE cus

### Results

- **Before Fine-Tuning:** Inconsistent and verbose outputs.
- **After Fine-Tuning:** More accurate and well-structured SQL queries.

## **8. Save the Adapters**

LoRA adapters are tiny (a few MB). You save *only* the adapter, not the whole base model.

In [21]:
ADAPTER_DIR = "./tinyllama-sql-lora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import os
total = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f"Adapter size on disk: {total/1024**2:.2f} MB")

Adapter size on disk: 20.67 MB


## **Loading the adapter later (for reference)**

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype=torch.float16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base, "./tinyllama-sql-lora-adapter")
tokenizer = AutoTokenizer.from_pretrained("./tinyllama-sql-lora-adapter")
```


### Limitations and Future Improvements

This project demonstrates LoRA-based fine-tuning for Text-to-SQL generation using a lightweight language model and a reduced training dataset. While the fine-tuned model generates more structured SQL queries, performance can be further improved through larger models, additional training data, and extended training epochs.

Future work includes training on the full dataset, evaluating query accuracy, and experimenting with larger LLMs such as Qwen or Llama-based models.

### Final Model Evaluation

The fine-tuned model is tested on a sample employee database schema. Given a natural language query, the model generates the corresponding SQL statement.

In [32]:
schema = """
CREATE TABLE employees (
    id INT,
    name TEXT,
    department TEXT,
    salary INT
);
"""

question = "Find employees earning more than 50000"

prompt = build_prompt(schema, question)

print(generate(prompt))

SELECT COUNT(id) FROM employees WHERE salary > 50000
